# Cantilever: solid beam coupled to a frame beam

Same `node_to_surface` idea as `example_column_nodeToSurface.ipynb`, but
here the master reference node is not just a fixity point — it is the
starting node of a **real frame element** that continues the beam
past the solid region.

```
       x=0                          x=L/2                       x=L
        ┬──────────────────────┬                                  
        │                      │                                 
  base  │   solid (tet4)       │ ---|-----|-----|-----|--- tip  
        │                      │   frame (elasticBeamColumn)   
        └──────────────────────┘                                  
       fixed                 ^ interface point                   
                             |                                   
                       node_to_surface                           
                   (master = interface point,                    
                    slave = solid end face)                      
```

The solid end face and the interface point are geometrically disjoint
— Gmsh meshes them as separate entities. `node_to_surface` bridges the
gap: it creates a phantom 6-DOF node for every mesh node on the end
face, rigid-links them to the interface point, and equal-DOFs each
phantom back to its original solid node in translations only. The
frame beam element then picks up from the interface point and carries
the load out to the tip.

We build the whole thing in apeGmsh, emit to OpenSees, run a single
static step, compare to the Euler–Bernoulli tip deflection, and
visualise the deformed shape in the apeGmsh Qt/VTK viewer.

In [1]:
from apeGmsh import apeGmsh, Results
import numpy as np

# ---- Geometry [mm] ------------------------------------------------
L_solid = 1000.0    # solid region length
L_frame = 1000.0    # frame region length
L_total = L_solid + L_frame
b       = 200.0     # cross-section width  (Y)
h       = 200.0     # cross-section depth  (Z)
lc      = 100.0     # mesh size

# ---- Material [MPa, -] --------------------------------------------
E  = 200_000.0
nu = 0.3
G  = E / (2 * (1 + nu))

# ---- Cross-section properties (square 200 x 200) ------------------
A   = b * h
Iy  = b * h**3 / 12.0       # bending about local y (in xz plane)
Iz  = h * b**3 / 12.0       # bending about local z (in xy plane)
J   = Iy + Iz               # rough torsional constant

# ---- Loading [N] --------------------------------------------------
P_tip = -5000.0             # tip force in the Z direction

print(f'L_total = {L_total} mm    cross-section = {b:.0f} x {h:.0f} mm')
print(f'EI_y    = {E*Iy:.3e} N*mm^2')

L_total = 2000.0 mm    cross-section = 200 x 200 mm
EI_y    = 2.667e+13 N*mm^2


## 1. Build the geometry

The solid half is an axis-aligned box centred on the X axis so its
end-face centroid is at `(L_solid, 0, 0)` — that is exactly where we
place the interface point. The frame half is a single `add_line`
between the interface and tip points; Gmsh will mesh it as a chain of
`line2` elements.

Labels / physical groups we create here:

| name | what | used for |
|------|------|----------|
| `pg_solid` | volume of the box | solid tet nodes / elements |
| `pg_frame` | the frame line entity | frame chain nodes / `line2` elements |
| `interface` | 0-D point at the master location | `node_to_surface` master |
| `tip` | 0-D point at the free end | tip load / monitored node |
| `base_face` | solid face at `x=0` | fixed support |
| `end_face` | solid face at `x=L_solid` | `node_to_surface` slave |

In [2]:
m = apeGmsh(model_name='cant_solid_frame', verbose=False)
m.begin()

# Solid half — centred on the X axis.
m.model.geometry.add_box(
    0, -b/2, -h/2, L_solid, b, h, label='solid_beam')
m.model.selection.select_volumes().to_physical(name='pg_solid')

# Reference points for the frame endpoints.
p_if  = m.model.geometry.add_point(L_solid, 0, 0, lc=lc, label='interface')
p_tip = m.model.geometry.add_point(L_total, 0, 0, lc=lc, label='tip')

# Frame half — one straight line meshed as line2.
line_tag = m.model.geometry.add_line(p_if, p_tip, label='frame_beam')
m.physical.add_curve(tags=[line_tag], name='pg_frame')

# Base fixity face (x=0) and end coupling face (x=L_solid).
# We use in_box with a thin slab instead of on_plane: on_plane matches
# any entity whose bounding box TOUCHES the plane, which incorrectly
# pulls in the four side faces of the box (they span x=0..L_solid).
# in_box requires the entity to be fully contained, which excludes them.
INF = max(b, h) * 2
m.model.selection.select_surfaces(
    in_box=(-0.1, -INF, -INF, 0.1, INF, INF)
).to_physical('base_face')
m.model.selection.select_surfaces(
    in_box=(L_solid - 0.1, -INF, -INF, L_solid + 0.1, INF, INF)
).to_physical('end_face')

# THE coupling — interface point (6-DOF) to solid end face.
m.constraints.node_to_surface('interface', 'end_face')

# Tip load (resolved as a NodalLoadRecord on the tip label).
m.loads.point(target='tip', force_xyz=[0, 0, P_tip], moment_xyz=[0, 0, 0])

m.mesh.sizing.set_global_size(lc)
m.mesh.generation.generate(dim=3)

fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

print(f'total nodes: {len(fem.nodes.ids)}')
for g in fem.elements:
    print(f'  {g.type_name:6s} n={len(g)}')
print(f'base_face nodes: {len(fem.nodes.get_ids(target="base_face"))}')
print(f'end_face  nodes: {len(fem.nodes.get_ids(target="end_face"))}')

total nodes: 121
  line2  n=66
  tri3   n=212
  tet4   n=254
  point1 n=10
base_face nodes: 12
end_face  nodes: 12


## 2. Inspect the coupling

One `NodeToSurfaceRecord` should show up in `fem.nodes.constraints`,
with as many phantom nodes as there are mesh nodes on the end face.
Every phantom is a 6-DOF duplicate of a slave node, rigid-linked from
the interface master and equal-DOF'd back to its original slave in
translations only. The `rigid_link_groups()` and `equal_dofs()`
iterators are what we will feed into OpenSees later — they expand
the compound record into atomic pair constraints automatically.

In [3]:
nc = fem.nodes.constraints
print(f'{len(nc)} compound record(s) in fem.nodes.constraints')

for nts in nc.node_to_surfaces():
    print(f"\n  name          : {nts.name}")
    print(f"  master node   : {nts.master_node}")
    print(f"  slave count   : {len(nts.slave_nodes)}")
    print(f"  phantom count : {len(nts.phantom_nodes)}")
    print(f"  rigid_beam pairs : {len(nts.rigid_link_records)}")
    print(f"  equal_dof pairs  : {len(nts.equal_dof_records)}")

print(f'\n.pairs()   (flat atomic iterator): {sum(1 for _ in nc.pairs())} pair(s)')
print(f'.equal_dofs():                     {sum(1 for _ in nc.equal_dofs())} pair(s)')

1 compound record(s) in fem.nodes.constraints

  name          : interface → end_face
  master node   : 9
  slave count   : 12
  phantom count : 12
  rigid_beam pairs : 12
  equal_dof pairs  : 12

.pairs()   (flat atomic iterator): 24 pair(s)
.equal_dofs():                     12 pair(s)


## 3. Emit the OpenSees model

Strategy:

1. Create the base model with `ndf=3`. The solid tet nodes only need
   translations.
2. Override `-ndf 6` node-by-node for:
   * the frame chain (`pg_frame`) — beam elements need rotations
   * the phantom nodes created by `node_to_surface`
3. Emit materials, solid tet elements, then the beam section and
   `elasticBeamColumn` elements.
4. Emit the coupling constraints (`rigid_link_groups()` → `rigidLink`,
   `equal_dofs()` → `equalDOF`).
5. Fix the base face, apply the tip load, run one static step.

In [4]:
import openseespy.opensees as ops

# Model is 6-DOF by default so geomTransf / elasticBeamColumn work.
# Solid tet nodes are marked ndf=3 via a per-node override below.
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)
ops.timeSeries('Linear', 1)

# --- 1. Solid nodes (ndf = 3 override) -----------------------------
n_solid = 0
for nid, xyz in fem.nodes.get(target='pg_solid'):
    ops.node(nid, *xyz, '-ndf', 3)
    n_solid += 1

# --- 2. Frame-chain nodes (ndf = 6, model default) ----------------
# target='pg_frame' resolves to all nodes on the line2 element chain,
# which already includes the interface and tip endpoint nodes.
frame_node_ids = []
for nid, xyz in fem.nodes.get(target='pg_frame'):
    ops.node(nid, *xyz)
    frame_node_ids.append(int(nid))

# --- 3. Phantom nodes (ndf = 6, model default) ---------------------
for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

print(f'solid nodes (ndf=3): {n_solid}')
print(f'frame nodes (ndf=6): {len(frame_node_ids)}')
print(f'phantom nodes (ndf=6): {len(fem.nodes.constraints.phantom_nodes())}')

solid nodes (ndf=3): 110
frame nodes (ndf=6): 11
phantom nodes (ndf=6): 12


In [5]:
# --- 4. Solid material + tet elements ------------------------------
mat_tag = 1
ops.nDMaterial('ElasticIsotropic', mat_tag, E, nu)

n_tets = 0
for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', eid, *conn, mat_tag)
        n_tets += 1

# --- 5. Frame section + beam elements ------------------------------
# geomTransf: local x = beam axis (global X), vecxz = global Z,
# so local y = global Y, local z = global Z. A tip load in -Z bends
# the beam about local y → uses Iy.
transf_tag = 1
ops.geomTransf('Linear', transf_tag, 0.0, 0.0, 1.0)

n_beams = 0
# Get only the line2 elements that belong to the frame line (pg_frame),
# not every line2 in the mesh (the solid box has line2 edges too).
for group in fem.elements.get(target='pg_frame'):
    if group.type_name != 'line2':
        continue
    for eid, conn in group:
        ni, nj = int(conn[0]), int(conn[1])
        ops.element('elasticBeamColumn', eid, ni, nj,
                    A, E, G, J, Iy, Iz, transf_tag)
        n_beams += 1

print(f'solid elements (tet4):          {n_tets}')
print(f'frame elements (elasticBeamColumn): {n_beams}')

solid elements (tet4):          254
frame elements (elasticBeamColumn): 10


In [6]:
# --- 6. Coupling: rigid links + equal DOFs -------------------------
# rigid_link_groups() yields (master, [slaves]) — one master per
# NodeToSurfaceRecord. equal_dofs() yields flat NodePairRecord objects.
n_rl = 0
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', master, slave)
        n_rl += 1

n_ed = 0
for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(pair.master_node, pair.slave_node, *pair.dofs)
    n_ed += 1

print(f'rigid links emitted: {n_rl}')
print(f'equal DOFs emitted:  {n_ed}')

# --- 7. Fix the base face ------------------------------------------
base_ids = list(int(n) for n in fem.nodes.get_ids(target='base_face'))
for nid in base_ids:
    ops.fix(nid, 1, 1, 1)  # solid nodes are ndf=3 — only 3 flags
print(f'base nodes fixed: {len(base_ids)}')

# --- 8. Tip load ----------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)
print(f'loads applied: {sum(1 for _ in fem.nodes.loads.by_kind("nodal"))}')

rigid links emitted: 12
equal DOFs emitted:  12
base nodes fixed: 12
loads applied: 1


In [7]:
# --- 9. One linear static step -------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 1.0)
ops.analysis('Static')

ok = ops.analyze(1)
assert ok == 0, f'analysis failed: {ok}'
print('analysis converged')

# Tip displacement (node target = 'tip' resolves to the 0-D point id).
tip_id = int(fem.nodes.get_ids(target='tip')[0])
tip_disp = ops.nodeDisp(tip_id)
print(f'\ntip node {tip_id} displacement:')
print(f'  ux = {tip_disp[0]:+.5e} mm')
print(f'  uy = {tip_disp[1]:+.5e} mm')
print(f'  uz = {tip_disp[2]:+.5e} mm')

analysis converged

tip node 10 displacement:
  ux = +1.58608e-06 mm
  uy = -2.43350e-03 mm
  uz = -2.87485e-01 mm


## 4. Sanity check — Euler–Bernoulli cantilever

For a linear tip-loaded cantilever of length `L`,

$$
\delta_{\mathrm{EB}} \;=\; \frac{P L^{3}}{3 E I}
$$

We expect the FEM result to be **stiffer** (smaller deflection) than the
closed-form value for two reasons:

1. **Linear tet4 is volumetrically over-stiff in bending.** The 4-node
   tetrahedron can only represent constant strain per element, so the
   coarse solid half is substantially stiffer than a pure Euler–Bernoulli
   beam of the same geometry. Refining the mesh (or switching to
   quadratic tet10 / hex8 elements) closes the gap.
2. **The cross-section is short and stocky** (h/L = 0.1), so shear
   deformation is not negligible — Timoshenko theory would already sit
   below the Euler–Bernoulli answer even with a perfect mesh.

A ratio of ~0.5–0.7 at `lc=100` is typical and does not indicate a
coupling error; it means the tet4 half of the beam is bearing load as
expected. What we really want to verify is that the deflection is
**non-trivial** (the coupling is transferring load end-to-end) and that
**uz dominates** (bending direction is correct).

In [8]:
delta_EB = abs(P_tip) * L_total**3 / (3.0 * E * Iy)
uz_fem   = tip_disp[2]

print(f'Euler-Bernoulli (pure beam, whole length):')
print(f'  delta_EB = P*L^3 / (3*E*Iy) = {delta_EB:+.5e} mm')
print()
print(f'FEM tip deflection (solid + frame):')
print(f'  uz_fem   = {uz_fem:+.5e} mm')
print()
print(f'ratio  FEM / EB = {uz_fem / (-delta_EB):+.4f}  '
      f'(1.0 = exact match, >1 = softer than EB)')

Euler-Bernoulli (pure beam, whole length):
  delta_EB = P*L^3 / (3*E*Iy) = +5.00000e-01 mm

FEM tip deflection (solid + frame):
  uz_fem   = -2.87485e-01 mm

ratio  FEM / EB = +0.5750  (1.0 = exact match, >1 = softer than EB)


## 5. Visualise the deformed shape

Build an `(N, 3)` displacement array in `fem.nodes.ids` order and pass
it to `Results.from_fem`. The apeGmsh viewer will show the full
solid + frame mesh with a `Displacement` vector field — toggle the
**Show Deformed** control to animate the bending.

Phantom nodes are *not* part of the visualised mesh (they exist only
to carry the constraint chain), so the viewer shows a clean solid +
frame without extra clutter.

## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [11]:
from apeGmsh import workdir
OUT = workdir()
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=6)

results_capture = OUT / "capture.h5"
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=6) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 7 ---
    import openseespy.opensees as ops

    # Model is 6-DOF by default so geomTransf / elasticBeamColumn work.
    # Solid tet nodes are marked ndf=3 via a per-node override below.
    ops.wipe()
    ops.model('basic', '-ndm', 3, '-ndf', 6)
    ops.timeSeries('Linear', 1)

    # --- 1. Solid nodes (ndf = 3 override) -----------------------------
    n_solid = 0
    for nid, xyz in fem.nodes.get(target='pg_solid'):
        ops.node(nid, *xyz, '-ndf', 3)
        n_solid += 1

    # --- 2. Frame-chain nodes (ndf = 6, model default) ----------------
    # target='pg_frame' resolves to all nodes on the line2 element chain,
    # which already includes the interface and tip endpoint nodes.
    frame_node_ids = []
    for nid, xyz in fem.nodes.get(target='pg_frame'):
        ops.node(nid, *xyz)
        frame_node_ids.append(int(nid))

    # --- 3. Phantom nodes (ndf = 6, model default) ---------------------
    for nid, xyz in fem.nodes.constraints.phantom_nodes():
        ops.node(nid, *xyz)

    print(f'solid nodes (ndf=3): {n_solid}')
    print(f'frame nodes (ndf=6): {len(frame_node_ids)}')
    print(f'phantom nodes (ndf=6): {len(fem.nodes.constraints.phantom_nodes())}')
    # --- copied from cell 8 ---
    # --- 4. Solid material + tet elements ------------------------------
    mat_tag = 1
    ops.nDMaterial('ElasticIsotropic', mat_tag, E, nu)

    n_tets = 0
    for group in fem.elements.get(element_type='tet4'):
        for eid, conn in group:
            ops.element('FourNodeTetrahedron', eid, *conn, mat_tag)
            n_tets += 1

    # --- 5. Frame section + beam elements ------------------------------
    # geomTransf: local x = beam axis (global X), vecxz = global Z,
    # so local y = global Y, local z = global Z. A tip load in -Z bends
    # the beam about local y → uses Iy.
    transf_tag = 1
    ops.geomTransf('Linear', transf_tag, 0.0, 0.0, 1.0)

    n_beams = 0
    # Get only the line2 elements that belong to the frame line (pg_frame),
    # not every line2 in the mesh (the solid box has line2 edges too).
    for group in fem.elements.get(target='pg_frame'):
        if group.type_name != 'line2':
            continue
        for eid, conn in group:
            ni, nj = int(conn[0]), int(conn[1])
            ops.element('elasticBeamColumn', eid, ni, nj,
                        A, E, G, J, Iy, Iz, transf_tag)
            n_beams += 1

    print(f'solid elements (tet4):          {n_tets}')
    print(f'frame elements (elasticBeamColumn): {n_beams}')
    # --- copied from cell 9 ---
    # --- 6. Coupling: rigid links + equal DOFs -------------------------
    # rigid_link_groups() yields (master, [slaves]) — one master per
    # NodeToSurfaceRecord. equal_dofs() yields flat NodePairRecord objects.
    n_rl = 0
    for master, slaves in fem.nodes.constraints.rigid_link_groups():
        for slave in slaves:
            ops.rigidLink('beam', master, slave)
            n_rl += 1

    n_ed = 0
    for pair in fem.nodes.constraints.equal_dofs():
        ops.equalDOF(pair.master_node, pair.slave_node, *pair.dofs)
        n_ed += 1

    print(f'rigid links emitted: {n_rl}')
    print(f'equal DOFs emitted:  {n_ed}')

    # --- 7. Fix the base face ------------------------------------------
    base_ids = list(int(n) for n in fem.nodes.get_ids(target='base_face'))
    for nid in base_ids:
        ops.fix(nid, 1, 1, 1)  # solid nodes are ndf=3 — only 3 flags
    print(f'base nodes fixed: {len(base_ids)}')

    # --- 8. Tip load ----------------------------------------------------
    ops.pattern('Plain', 1, 1)
    for load in fem.nodes.loads.by_kind('nodal'):
        fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
        mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
        ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)
    print(f'loads applied: {sum(1 for _ in fem.nodes.loads.by_kind("nodal"))}')
    # --- copied from cell 10 ---
    # --- 9. One linear static step -------------------------------------
    ops.constraints('Penalty', 1e15, 1e15)
    ops.numberer('RCM')
    ops.system('UmfPack')
    ops.test('NormDispIncr', 1.0e-6, 10)
    ops.algorithm('Linear')
    ops.integrator('LoadControl', 1.0)
    ops.analysis('Static')

    ok = ops.analyze(1)
    cap.step(t=ops.getTime())
    assert ok == 0, f'analysis failed: {ok}'
    print('analysis converged')

    # Tip displacement (node target = 'tip' resolves to the 0-D point id).
    tip_id = int(fem.nodes.get_ids(target='tip')[0])
    tip_disp = ops.nodeDisp(tip_id)
    print(f'\ntip node {tip_id} displacement:')
    print(f'  ux = {tip_disp[0]:+.5e} mm')
    print(f'  uy = {tip_disp[1]:+.5e} mm')
    print(f'  uz = {tip_disp[2]:+.5e} mm')
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


solid nodes (ndf=3): 110


frame nodes (ndf=6): 11
phantom nodes (ndf=6): 12
solid elements (tet4):          254
frame elements (elasticBeamColumn): 10
rigid links emitted: 12
equal DOFs emitted:  12
base nodes fixed: 12
loads applied: 1
analysis converged

tip node 10 displacement:
  ux = +1.58608e-06 mm
  uy = -2.43350e-03 mm
  uz = -2.87485e-01 mm
capture -> example_cantilever_solid_to_frame_capture.h5 (113.3 KB)


In [12]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess, non-blocking).
# Set APEGMSH_SKIP_VIEWER=1 in the environment to skip the GUI under nbconvert / CI.
from apeGmsh.results import Results
results = Results.from_native(results_capture)
results.viewer(blocking=False)


[skip viewer] APEGMSH_SKIP_VIEWER set
